# 00 — Prepare publication data package

Run this notebook **once on Craig's Ubuntu machine**.

It reads the full development files from the ExtremeSSD and creates a deliberately
minimal publication dataset in a separate directory. The output contains no wreck
identifiers or wreck coordinates.

The downstream analysis notebooks are repository-relative and do not depend on this
SSD path; once the publication directory has been prepared it can be copied to Linux,
macOS or Windows.


In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import xarray as xr

# ------------------------------------------------------------------
# User-editable paths
# ------------------------------------------------------------------

SSD = Path("/media/syms/ExtremeSSD")
SOURCE_ROOT = SSD / "ShipwreckENDURE"
PUBLIC_ROOT = SSD / "ShipwreckENDURE_Publication"

SOURCE_REGIONAL = SOURCE_ROOT / "ABMsubset.nc"
SOURCE_WRECKS = SOURCE_ROOT / "wreck_agent_decay_for_gam.parquet"

PUBLIC_DATA = PUBLIC_ROOT / "data"
PUBLIC_MODEL_INPUTS = PUBLIC_DATA / "model_inputs"
PUBLIC_CALIBRATION = PUBLIC_DATA / "calibration"

for directory in (
    PUBLIC_ROOT,
    PUBLIC_DATA,
    PUBLIC_MODEL_INPUTS,
    PUBLIC_CALIBRATION,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Source:", SOURCE_ROOT)
print("Publication package:", PUBLIC_ROOT)


Source: /media/syms/ExtremeSSD/ShipwreckENDURE
Publication package: /media/syms/ExtremeSSD/ShipwreckENDURE_Publication


## 1. Create the minimal regional model input

The core deterioration engine needs long-term mean environmental fields. The
manuscript-output notebook additionally uses a small number of variance fields for
Supplement 4. These are retained explicitly below.

No raw Copernicus/EMODnet time series are copied into the publication package.


In [2]:
model_vars = [
    # Core environmental model inputs
    "bottomT_mean",
    "thetao_mean",
    "o2_mean",
    "po4_mean",
    "no3_mean",
    "nppv_mean",
    "so_mean",
    "ph_mean",
    "fe_mean",
    "chl_mean",
    "uo_mean",
    "vo_mean",
    "current_speed_mean",
    "seabed_orbital_velocity_mean",
    "deptho_mean",
    "substrate_soft_hard",

    # Additional fields used directly in Supplement 4
    "bottomT_var_total",
    "so_var_total",
    "so_var_interannual",
    "chl_var_total",
    "chl_var_interannual",
    "VHM0_mean_mean",
    "VHM0_mean_var_total",
    "VHM0_p95_mean",
]

publication_nc = PUBLIC_MODEL_INPUTS / "publication_model_inputs.nc"

with xr.open_dataset(SOURCE_REGIONAL) as ds:
    missing = [v for v in model_vars if v not in ds.data_vars]

    if missing:
        raise KeyError(
            "The source ABMsubset.nc is missing required publication variables:\n"
            + "\n".join(missing)
        )

    publication_ds = ds[model_vars].load()

    # Retain useful global provenance, but identify this as the publication subset.
    publication_ds.attrs.update(ds.attrs)
    publication_ds.attrs["publication_subset"] = (
        "Minimal derived environmental input for the ENDURE shipwreck "
        "deterioration manuscript."
    )
    publication_ds.attrs["source_file"] = "ABMsubset.nc"
    publication_ds.attrs["contains_restricted_wreck_coordinates"] = "no"

    
publication_ds = publication_ds.sel(
    latitude=slice(54.0, 59.0),
    longitude=slice(2.4166667, 22.0)
)


publication_ds.to_netcdf(publication_nc)

print("Written:", publication_nc)
print("Dimensions:", dict(publication_ds.sizes))
print("Variables:", len(publication_ds.data_vars))
print("Size (MiB):", publication_nc.stat().st_size / 1024**2)


Written: /media/syms/ExtremeSSD/ShipwreckENDURE_Publication/data/model_inputs/publication_model_inputs.nc
Dimensions: {'latitude': 61, 'longitude': 236}
Variables: 24
Size (MiB): 2.604490280151367


## 2. Create the anonymised GAM calibration dataset

The production GAM originally filtered complete cases using the contractor-listing
field as well as the four analytical variables. To reproduce the **same rows** while
not releasing identifying information, this notebook applies the original filter and
then exports only the four variables used by the fitted GAM.

The published CSV therefore contains no wreck name, site identifier, latitude,
longitude, survey filename, notes or provenance fields.


In [3]:
filter_cols = [
    "JD.Contractor.Listing",
    "Obs",
    "chronological_age",
    "log_CR_physics",
    "log_CR_bio",
]

public_cols = [
    "Obs",
    "chronological_age",
    "log_CR_physics",
    "log_CR_bio",
]

wrecks = pd.read_parquet(SOURCE_WRECKS, columns=filter_cols)

cal = wrecks.loc[
    wrecks[filter_cols].notna().all(axis=1)
    & (wrecks["Obs"] > 0)
    & (wrecks["chronological_age"] > 0),
    public_cols,
].copy()

cal["Obs"] = cal["Obs"].astype(int)
cal = cal.reset_index(drop=True)

publication_calibration = PUBLIC_CALIBRATION / "wreck_calibration.csv"
cal.to_csv(publication_calibration, index=False)

print("Written:", publication_calibration)
print("Rows:", len(cal))
print("Columns:", list(cal.columns))
print("Age range:", cal["chronological_age"].min(), "to", cal["chronological_age"].max())
print("Observed classes:", sorted(cal["Obs"].unique().tolist()))


Written: /media/syms/ExtremeSSD/ShipwreckENDURE_Publication/data/calibration/wreck_calibration.csv
Rows: 124
Columns: ['Obs', 'chronological_age', 'log_CR_physics', 'log_CR_bio']
Age range: 16.0 to 216.0
Observed classes: [1, 2, 3, 4]


## 3. Privacy / leakage checks

These are deliberately conservative. The public calibration table should contain
exactly four columns and none of the common identifying/location field names.


In [4]:
expected_public_cols = {
    "Obs",
    "chronological_age",
    "log_CR_physics",
    "log_CR_bio",
}

if set(cal.columns) != expected_public_cols:
    raise RuntimeError(
        "Unexpected columns in public calibration table: "
        f"{list(cal.columns)}"
    )

for forbidden in (
    "lat",
    "lon",
    "longitude",
    "latitude",
    "listing",
    "name",
    "notes",
    "survey_date",
    "site",
    "wreck_id",
):
    hits = [c for c in cal.columns if forbidden.lower() in c.lower()]
    if hits:
        raise RuntimeError(f"Potential identifying field retained: {hits}")

print("Privacy check passed.")


Privacy check passed.


## 4. Write a machine-readable manifest

The manifest records exactly which variables were retained and basic file hashes.
This is useful when the package is eventually frozen in Zenodo.


In [5]:
def sha256(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

manifest = {
    "publication_model_inputs": {
        "file": "data/model_inputs/publication_model_inputs.nc",
        "variables": model_vars,
        "dimensions": {k: int(v) for k, v in publication_ds.sizes.items()},
        "sha256": sha256(publication_nc),
    },
    "wreck_calibration": {
        "file": "data/calibration/wreck_calibration.csv",
        "rows": int(len(cal)),
        "columns": list(cal.columns),
        "sha256": sha256(publication_calibration),
    },
}

manifest_file = PUBLIC_ROOT / "data_manifest.json"
manifest_file.write_text(json.dumps(manifest, indent=2))

print("Written:", manifest_file)
print("\nPublication data preparation complete.")
print("Next: copy the repository src/ and analysis/ directories into")
print(PUBLIC_ROOT)
print("then run analysis notebooks 01 -> 04 in order.")


Written: /media/syms/ExtremeSSD/ShipwreckENDURE_Publication/data_manifest.json

Publication data preparation complete.
Next: copy the repository src/ and analysis/ directories into
/media/syms/ExtremeSSD/ShipwreckENDURE_Publication
then run analysis notebooks 01 -> 04 in order.
